# 04 - Feature Selection

This notebook demonstrates the feature selection pipeline:
1. Load preprocessed train/test data
2. Run Variance Threshold → Mutual Information → Binary PSO on training data ONLY
3. Transform test data using fitted selectors
4. Save selected features


In [17]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
import config
from src.io import save_dataframe, save_table, logger
# from src.feature_selection import run_feature_selection, transform_selected
from src.visualization import setup_style, plot_feature_importance
setup_style()

## Step 1: Load Preprocessed Data

In [18]:
X_train = pd.read_csv(config.PROCESSED_DIR / "X_train_preprocessed.csv")
X_test = pd.read_csv(config.PROCESSED_DIR / "X_test_preprocessed.csv")
y_train = pd.read_csv(config.PROCESSED_DIR / "y_train.csv").iloc[:, 0]
y_test = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").iloc[:, 0]

print(f"Train shape: {X_train.shape}, positives={int(y_train.sum())}")
print(f"Test shape:  {X_test.shape}, positives={int(y_test.sum())}")

Train shape: (343, 19019), positives=46
Test shape:  (86, 19019), positives=12


## Step 2: Run Feature Selection Pipeline (Train ONLY)

Pipeline: **Variance Threshold → Mutual Information → Binary PSO**

All steps are fitted exclusively on `X_train` / `y_train`.

In [19]:
# fitted_selector, final_features = run_feature_selection(
#     X_train,
#     y_train,
#     variance_threshold=config.VARIANCE_THRESHOLD,
#     mi_top_k=config.MI_TOP_K,
#     pso_final_k=config.PSO_FINAL_K,
#     run_pso=True,
#     random_state=config.RANDOM_STATE,
# )

# print(f"\nVariance features: {len(fitted_selector['variance_features'])}")
# print(f"MI features:       {len(fitted_selector['mi_features'])}")
# print(f"Final PSO features: {len(final_features)}")
# ============================================================================
# RUN THE NEW 3-LAYER FEATURE SELECTION PIPELINE (FIXED)
# ============================================================================


In [20]:
# Cell 1: Setup & Imports
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import joblib
import config
from src.io import logger
from src.feature_selection import run_3layer_feature_selection

# Load preprocessed training data
X_train = pd.read_csv(config.PROCESSED_DIR / "X_train_preprocessed.csv", index_col=0)
y_train_df = pd.read_csv(config.PROCESSED_DIR / "y_train.csv")
y_train = y_train_df.iloc[:, 0] if len(y_train_df.columns) == 1 else y_train_df['BCR']

logger.info(f"Training data loaded: {X_train.shape}, Positives: {int(y_train.sum())}")

# Cell 2: Run 3-Layer Feature Selection Pipeline
print("🚀 Starting 3-Layer Feature Selection Pipeline...")
print(f"   Target Features: {config.PSO_FINAL_K} (Recommended: 40-45)")

try:
    # Comprehensive clinical keywords to prevent leakage into Layer 1
    clinical_keywords = [
        'gleason', 'margin', 'lymph', 'tumor stage', 'psa', 
        'bone scan', 'cause of death', 'ct scan', 'primary therapy',
        'age', 'race', 'ethnicity', 'weight', 'height',
        'mri', 'icd-o', 'histology', 'patient primary', 'diagnosis',
        'year cancer', 'radical prostatectomy', 'adjuvant', 'radiation',
        'hormone', 'chemotherapy', 'surgery', 'metastasis', 'recurrence'
    ]
    
    auto_clinical_cols = [
        c for c in X_train.columns 
        if any(kw.lower() in c.lower() for kw in clinical_keywords)
    ]
    
    engineered_keywords = ['pathway_score', '_score', 'risk', 'total', 'ratio', 'balance']
    auto_clinical_cols += [
        c for c in X_train.columns 
        if any(kw.lower() in c.lower() for kw in engineered_keywords) and c not in auto_clinical_cols
    ]
    
    print(f"   Detected {len(auto_clinical_cols)} clinical/engineered columns to exclude from Layer 1.")
    
    fitted_l1, final_features = run_3layer_feature_selection(
        X_train=X_train,
        y_train=y_train,
        clinical_cols=auto_clinical_cols,
        run_pso=True,
        random_state=config.RANDOM_STATE
    )

    print(f"\n✅ Pipeline Complete!")
    print(f"   Layer 1 (Genes):      {len(fitted_l1['mi_features'])} MI-selected genes")
    print(f"   Final Total Features: {len(final_features)}")
    print(f"   Target Range Met:     {'YES' if 40 <= len(final_features) <= 45 else 'NO'}")

    # Save artifacts for Notebook 05
    joblib.dump(fitted_l1, config.MODELS_DIR / "fitted_layer1_selector.joblib")
    pd.DataFrame({"feature": final_features}).to_csv(
        config.TABLES_DIR / "selected_features_final.csv", index=False
    )
    print(f"   Saved 'fitted_layer1_selector.joblib' and 'selected_features_final.csv'")

except ValueError as e:
    if "feature names should match" in str(e).lower():
        print(f"\n❌ CRITICAL ERROR: Column mismatch detected in Layer 1 transform.")
        print(f"   Error Details: {e}")
        print("\n💡 FIX: Ensure X_train contains ALL original gene columns.")
    else:
        raise

2026-08-31 14:23:23 | INFO     | prostate_bcr | Training data loaded: (343, 19018), Positives: 46


🚀 Starting 3-Layer Feature Selection Pipeline...
   Target Features: 30 (Recommended: 40-45)
   Detected 132 clinical/engineered columns to exclude from Layer 1.


2026-08-31 14:23:54 | INFO     | prostate_bcr | Layer 1 - Selected 200 genes from 18886 raw genes



❌ CRITICAL ERROR: Column mismatch detected in Layer 1 transform.
   Error Details: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Tumor Other Histologic Subtype_25-30% ductal component
- Tumor Other Histologic Subtype_Adenocarcinoma prostate with prominent ductal differentiation identified
- Tumor Other Histologic Subtype_Mixed
- Tumor Other Histologic Subtype_Mixed ductal (65%) and Acinar
- Tumor Other Histologic Subtype_Prostate Adenocarcinoma, Not Otherwised Specified, with ductal featues
- ...


💡 FIX: Ensure X_train contains ALL original gene columns.


## Step 3: Display Selected Features

In [21]:
# selected_df = pd.DataFrame({
#     "rank": range(1, len(final_features) + 1),
#     "feature": final_features,
# })

# print("Selected features:")
# print(selected_df.to_string(index=False))

## Step 4: Mutual Information Scores (Top Features)

In [22]:
# mi_scores = fitted_selector["mi_scores"]
# top_mi = mi_scores.head(20)

# fig, ax = plt.subplots(figsize=(10, 7))
# top_mi.sort_values().plot(kind="barh", ax=ax, color="#3C5488")
# ax.set_xlabel("Mutual Information Score")
# ax.set_title("Top 20 Features by Mutual Information")
# ax.tick_params(axis="y", labelsize=9)
# plt.tight_layout()

# from src.io import save_figure
# save_figure(fig, "feature_selection_mi_scores.png")
# plt.show()

## Step 5: Transform Test Data Using Fitted Selector

> **Leakage Prevention:** Test data is transformed using selectors
> that were fitted ONLY on training data.

In [23]:
# X_train_selected = transform_selected(X_train, fitted_selector, final_features)
# X_test_selected = transform_selected(X_test, fitted_selector, final_features)

# print(f"Train selected shape: {X_train_selected.shape}")
# print(f"Test selected shape:  {X_test_selected.shape}")

## Step 6: Save Selected Features and Transformed Data

In [24]:
# # Save feature list
# save_table(selected_df, "selected_features.csv", index=False)

# # Save transformed data
# save_dataframe(X_train_selected, "X_train_selected.csv", index=False)
# save_dataframe(X_test_selected, "X_test_selected.csv", index=False)

# print("Saved:")
# print(f"  - selected_features.csv ({len(final_features)} features)")
# print(f"  - X_train_selected.csv ({X_train_selected.shape})")
# print(f"  - X_test_selected.csv ({X_test_selected.shape})")